#### **Base and Large model evaluation prior fine-tuning**

This notebook covers the evaluation of the project. It has two main purposes, both linked to the evaluation part of our project:

* the assessment of flan-t5-base and flan-t5-large on the test split at four precision levels : **FP32, BF16, INT8, INT4**, using Exact Match (EM) and F1.
* the build and explaination of the different functions that **evaluate.py** contain, ending with **evaluate_model()**, which is the function we'll use throughout the project to score a model.

In order to run this notebook from top to bottom, you need to have cloned first the github repo.

**Let us first assess the two models on different quantisation levels**:

In [1]:
!git clone https://github.com/Julianlls/DeepLearning.git
%cd DeepLearning
!pip install -q transformers datasets accelerate bitsandbytes

Cloning into 'DeepLearning'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 99 (delta 43), reused 83 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 98.17 KiB | 5.77 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/DeepLearning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.0 MB/s eta 0:00:00


In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.evaluate import evaluate_model
from src.preprocessing import load_processed
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

ds = load_processed()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/27000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

cuda


In [3]:
KWARGS = {
    "fp32": {},
    "bf16": {"torch_dtype": torch.bfloat16},
    "int8": {"quantization_config": BitsAndBytesConfig(load_in_8bit=True),
             "device_map": "auto"},
    "int4": {"quantization_config": BitsAndBytesConfig(load_in_4bit=True),
             "device_map": "auto"},
}

def load_model(model_name, precision_mode):

    if precision_mode not in KWARGS:
        raise ValueError(f"Unknown precision_mode: {precision_mode!r}")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        f"google/{model_name}", **KWARGS[precision_mode]
    )

    if precision_mode in ("fp32", "bf16"):
        model = model.to(device)

    return model

In [4]:
import logging
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

In [5]:
import pandas as pd

configs = [
    ("flan-t5-base",  "fp32"),
    ("flan-t5-base",  "bf16"),
    ("flan-t5-base",  "int8"),
    ("flan-t5-base",  "int4"),
    ("flan-t5-large", "fp32"),
    ("flan-t5-large", "bf16"),
    ("flan-t5-large", "int8"),
    ("flan-t5-large", "int4"),
]

rows = []

for model_name, precision_mode in configs:
    print(f"--- {model_name} / {precision_mode} ---")

    model = load_model(model_name, precision_mode)
    row, preds = evaluate_model(
        model, tokenizer, ds["test"],
        model_name=model_name,
        state="raw",
        precision_mode=precision_mode,
    )
    rows.append(row)
    print("\n",row,sep="")

    del model
    if device == 'mps': torch.mps.empty_cache()
    else: torch.cuda.empty_cache()

pd.DataFrame(rows)

--- flan-t5-base / fp32 ---


model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generating:   0%|          | 0/375 [00:00<?, ?it/s]


{'model_name': 'flan-t5-large', 'state': 'raw', 'precision_mode': 'int4', 'exact_match': 0.53033, 'precision': 0.66869, 'recall': 0.68061, 'f1': 0.64755, 'n_examples': 3000}


,model_name,state,precision_mode,exact_match,precision,recall,f1,n_examples
0,flan-t5-base,raw,fp32,0.42367,0.55871,0.55531,0.53332,3000
1,flan-t5-base,raw,bf16,0.42567,0.55996,0.55622,0.53420,3000
2,flan-t5-base,raw,int8,0.42200,0.55709,0.55543,0.53232,3000
3,flan-t5-base,raw,int4,0.40333,0.54037,0.52487,0.50944,3000
4,flan-t5-large,raw,fp32,0.54833,0.68418,0.69684,0.66391,3000
5,flan-t5-large,raw,bf16,0.54900,0.68548,0.69825,0.66502,3000
6,flan-t5-large,raw,int8,0.54900,0.68483,0.69842,0.66469,3000
7,flan-t5-large,raw,int4,0.53033,0.66869,0.68061,0.64755,3000


### **Results**

| Model | Precision | EM | token_precision | token_recall | F1 | F1 vs.FP32 |
|---|---|---|---|---|---|---|
| flan-t5-base | FP32 | 0.4237 | 0.5587 | 0.5553 | 0.5333 | - |
| flan-t5-base | BF16 | 0.4257 | 0.5600 | 0.5562 | 0.5342 | +0.001 |
| flan-t5-base | INT8 | 0.4220 | 0.5571 | 0.5554 | 0.5323 | -0.001 |
| flan-t5-base | INT4 | 0.4033 | 0.5404 | 0.5249 | 0.5094 | −0.024 |
| flan-t5-large | FP32 | 0.5483 | 0.6842 | 0.6968 | 0.6639 | - |
| flan-t5-large | BF16 | 0.5490 | 0.6855 | 0.6983 | 0.6650 | +0.001 |
| flan-t5-large | INT8 | 0.5490 | 0.6848 | 0.6984 | 0.6647 | +0.001 |
| flan-t5-large | INT4 | 0.5303 | 0.6687 | 0.6806 | 0.6476 | −0.016 |

#### **Interpretation prior fine-tuning**

* **BF16 and INT8 are free** for both models. BF16 and INT8 have +/- 0.001 F1 of the baseline FP32. We can see that BF16 has an even higher F1, on both models. Differences of such size likely reflect numerical noise, not a degredation due to reduced precision. Halving the weights with BF16 or quartering it with INT8 leaves answer quality unchanged on this task. 

* **INT4 is the first cost**. It is the only precision level where there is a drop, in both models: -0.024 F1 for the base (-4.5%) and -0.016 F1 for the large (-2.5%). The loss is measurable but still modest, meaning that INT4 keeps roughly 95% of the baseline quality at a quarter of the storage. 

* **The larger model absorb quantization better.** Base loses 4.5% of its F1 at INT4 while large only loses 2.5%. More importantly, large at INT4 has F1 of 0.648, which is still outperforming base at FP32 (F1 of 0.533). This means that under memory constraint, quantized larger model is a better choice than a full precision smaller one. 

* **EM is much lower than F1** by roughly 0.11-0.12 across all eight cells. This is expected for a generative model on an extractive task (nothing forces the model to reproduce the exact annotated and expected span). Moreover, important to keep in mind that F1 is order order insensitive, just comparing bags of tokens, while EM expect the exact same order.

Bear in mind that flan-t5's instruction tuning includes extractive QA examples, so these are not just "zero-shot" scores in the strict sence since the model has seen the task format, just not this dataset. The fine-tuned half of the grid will show whether the precision patterns observed here hold after task-specific fine-tuning.

#### **evaluate.py construction**
The rest of this notebooks walks through how each function in src.evaluate.py was built and verified.


We first need a function to normalize the text that we'll use on the target answer and the predicted text for our models

In [ ]:
import re
import string


def normalize_answer(s: str) -> str:

    def lower(text):
        return text.lower()

    def remove_punc(text):
        exclude = string.punctuation
        return "".join(ch for ch in text if ch not in exclude)

    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    return white_space_fix(remove_articles(remove_punc(lower(s))))

Let us now define the different function that we will use for evaluation.

First the Exact Match (EM) score:

In [ ]:
def exact_match_score(prediction: str, target: str) -> int:
    #1 if the two strings match after normalization, 0 otherwise.
    return int(normalize_answer(prediction) == normalize_answer(target))

In [ ]:
tests = [
    ("Paris", "Paris", 1),
    ("paris", "Paris", 1),
    ("The Norman conquest.", "Norman conquest", 1),
    ("Norman conquest of England","the Norman conquest", 0),
    ("1066", "in 1066", 0),
    ("", "Paris", 0),
    ("", "", 1),
]

for pred, target, expected in tests:
    got = exact_match_score(pred, target)
    flag = "ok" if got == expected else "MISMATCH"
    print(f"{pred!r:30} vs {target!r:22} -> {got}  (expected {expected})  {flag}")

'Paris'                        vs 'Paris'                -> 1  (expected 1)  ok
'paris'                        vs 'Paris'                -> 1  (expected 1)  ok
'The Norman conquest.'         vs 'Norman conquest'      -> 1  (expected 1)  ok
'Norman conquest of England'   vs 'the Norman conquest'  -> 0  (expected 0)  ok
'1066'                         vs 'in 1066'              -> 0  (expected 0)  ok
''                             vs 'Paris'                -> 0  (expected 0)  ok
''                             vs ''                     -> 1  (expected 1)  ok


Now the F1 score metric, which represent the harmonic mean of precision and recall

In [ ]:
from collections import Counter

def f1_score(prediction: str, target: str) -> dict:

    pred_tokens = normalize_answer(prediction).split()
    target_tokens = normalize_answer(target).split()

    # Empty-string guard: full credit only if both sides are empty.
    if len(pred_tokens) == 0 or len(target_tokens) == 0:
        score = float(pred_tokens == target_tokens)
        return {"precision": score, "recall": score, "f1": score}

    common = Counter(pred_tokens) & Counter(target_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    precision = num_same / len(pred_tokens)
    recall = num_same / len(target_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}

In [ ]:
tests = [
    ("Paris",                      "Paris"),
    ("1066",                       "in 1066"),
    ("I forgot to say hello you",  "hello you"),
    ("Norman conquest of England", "the Norman conquest"),
    ("new york new york",          "new york new york"),
    ("new new new york",           "new york"),
    ("completely wrong",           "Paris"),
    ("",                           "Paris"),
    ("you hello",                  "hello you"),
]

for pred, gold in tests:
    s = f1_score(pred, gold)
    em = exact_match_score(pred, gold)
    print(f"{pred!r:30} vs {gold!r:22} "
          f"P={s['precision']:.3f} R={s['recall']:.3f} F1={s['f1']:.3f}  EM={em}")

'Paris'                        vs 'Paris'                P=1.000 R=1.000 F1=1.000  EM=1
'1066'                         vs 'in 1066'              P=1.000 R=0.500 F1=0.667  EM=0
'I forgot to say hello you'    vs 'hello you'            P=0.333 R=1.000 F1=0.500  EM=0
'Norman conquest of England'   vs 'the Norman conquest'  P=0.500 R=1.000 F1=0.667  EM=0
'new york new york'            vs 'new york new york'    P=1.000 R=1.000 F1=1.000  EM=1
'new new new york'             vs 'new york'             P=0.500 R=1.000 F1=0.667  EM=0
'completely wrong'             vs 'Paris'                P=0.000 R=0.000 F1=0.000  EM=0
''                             vs 'Paris'                P=0.000 R=0.000 F1=0.000  EM=0
'you hello'                    vs 'hello you'            P=1.000 R=1.000 F1=1.000  EM=0


Our functions so far work for one example of target output and predicted output. We now need to make it work for a whole batch.

In [ ]:
def compute_metrics(predictions: list, references: list) -> dict:

    #Average EM, precision, recall and F1 over a batch
    #predictions: list of strings of all decoded model output per example
    #references:  list of strings of all target answer per example

    if len(predictions) != len(references):
        raise ValueError(
            f"Length mismatch: {len(predictions)} predictions "
            f"vs {len(references)} references"
        )

    em_total = 0.0
    precision_total = 0.0
    recall_total = 0.0
    f1_total = 0.0

    for pred, target in zip(predictions, references):
        em_total += exact_match_score(pred, target)
        scores = f1_score(pred, target)
        precision_total += scores["precision"]
        recall_total += scores["recall"]
        f1_total += scores["f1"]

    n = len(predictions)

    return {
        "exact_match": round(em_total / n,5),
        "precision": round(precision_total / n,5),
        "recall": round(recall_total / n,5),
        "f1": round(f1_total / n,5),
        "n_examples": n,
    }

In [ ]:
preds = [
    "Paris",
    "1066",
    "I forgot to say hello you",
    "completely wrong",
]
golds = [
    "Paris",
    "in 1066",
    "hello you",
    "Paris",
]

results = compute_metrics(preds, golds)
for k, v in results.items():
    print(f"{k:14} {v}")

exact_match    0.25
precision      0.58
recall         0.62
f1             0.54
n_examples     4


We now need to build the function that will generate the prediction with the model over a batch of examples

In [ ]:
import sys
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import MAX_GEN_TOKENS, EVAL_BATCH_SIZE, DATA_PROCESSED, RESULTS_DIR

MODEL_NAME = "google/flan-t5-base"


device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: {device}")

dataset = load_from_disk(DATA_PROCESSED)
print(dataset)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

device: mps
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 27000
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
test_set = dataset["test"]
sample = test_set.select(range(50))
print(f"full test set size: {len(test_set)} | sample size: {len(sample)}")

full test set size: 3000 | sample size: 50


In [ ]:
from tqdm.auto import tqdm

def generate_predictions(model, tokenizer, dataset,
                         batch_size=EVAL_BATCH_SIZE,
                         max_new_tokens=MAX_GEN_TOKENS) -> list:

    # Run greedy generation on a dataset and return the decoded answer strings

    model.eval()
    device = next(model.parameters()).device
    predictions = []

    for start in tqdm(range(0, len(dataset), batch_size), desc="generating"):
        batch = dataset[start:start + batch_size]

        inputs = tokenizer.pad(
            {
                "input_ids": batch["input_ids"],
                "attention_mask": batch["attention_mask"],
            },
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        predictions.extend(decoded)

    return predictions

In [ ]:
sample

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})

In [ ]:
preds = generate_predictions(model, tokenizer, sample, batch_size=10)

for i in range(len(sample)):
    print(f"Question: {sample[i]['question']}")
    print(f"target: {sample[i]['answers']['text'][0]}")
    print(f"pred: {preds[i]!r}\n")

generating:   0%|          | 0/5 [00:00<?, ?it/s]

Question: Where is the Hoppings funfair held?
target: Town Moor
pred: 'Newcastle'

Question: Which park in England has an alliterative name?
target: Hampstead Heath
pred: 'Town Moor'

Question: What is a soccer organization called in England?
target: Club
pred: 'Football Club'

Question: Where is the Town Moor?
target: Newcastle
pred: 'Newcastle'

Question: What makes Town Moor suitable to graze cattle on it?
target: green space
pred: "It is larger than London's famous Hyde Park and Hampstead Heath put together"

Question: Where do the owners of the cattle that graze in the Town Moor live?
target: Newcastle
pred: 'Newcastle'

Question: Which is possibly found in Town Moor ?
target: cattle
pred: 'cattle'

Question: Which actors are honorary freemen?
target: the Royal Shakespeare Company
pred: 'Bob Geldof, King Harald V of Norway, Bobby Robson, Alan Shearer, the late Nelson Mandela and the Royal Shakespeare Company'

Question: Who shares a name with an older type of transportation?
targe

In [ ]:
targets = [example["answers"]["text"][0] for example in sample]
print(compute_metrics(preds, targets))

{'exact_match': 0.28, 'precision': 0.5, 'recall': 0.53, 'f1': 0.48, 'n_examples': 50}


Finally, we need now to build the function that merges computes_metrics, and generate_predictions

In [ ]:
def evaluate_model(model, tokenizer, dataset,
                   model_name, state, precision_mode,
                   batch_size=EVAL_BATCH_SIZE,
                   max_new_tokens=MAX_GEN_TOKENS):

    predictions = generate_predictions(
        model, tokenizer, dataset,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
    )

    references = [example["text"][0] for example in dataset["answers"]]
    metrics = compute_metrics(predictions, references)

    results_row = {
        "model_name": model_name,
        "state": state,
        "precision_mode": precision_mode,
        **metrics,
    }

    return results_row, predictions

In [ ]:
row, preds = evaluate_model(
    model, tokenizer, sample,
    model_name="flan-t5-base",
    state="raw",
    precision_mode="fp32",
)

for k, v in row.items():
    print(f"{k:16} {v}")

generating:   0%|          | 0/7 [00:00<?, ?it/s]

model_name       flan-t5-base
state            raw
precision_mode   fp32
exact_match      0.28
precision        0.5
recall           0.53
f1               0.48
n_examples       50


Let us now perform the evaluation on the test dataset

In [ ]:
row, preds = evaluate_model(
    model, tokenizer, test_set,
    model_name="flan-t5-base",
    state="raw",
    precision_mode="fp32",
)

for k, v in row.items():
    print(f"{k:16} {v}")

generating:   0%|          | 0/375 [00:00<?, ?it/s]

model_name       flan-t5-base
state            raw
precision_mode   fp32
exact_match      0.42
precision        0.56
recall           0.56
f1               0.53
n_examples       3000
